In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import csv
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

# ---------------------------
# Load data
# ---------------------------
train_raw = pd.read_csv("train.csv")
test_raw  = pd.read_csv("test.csv")

FEATURE_COLS = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL']
TARGET_COL   = ['OT']

# ---------------------------
# Scale features + target
# ---------------------------
feature_scaler = MinMaxScaler()
target_scaler  = MinMaxScaler()

train_features_scaled = feature_scaler.fit_transform(train_raw[FEATURE_COLS])
train_target_scaled   = target_scaler.fit_transform(train_raw[TARGET_COL])

test_features_scaled  = feature_scaler.transform(test_raw[FEATURE_COLS])

# ---------------------------
# Convert to tensors (NO sequences)
# Shape: (batch, seq_len=1, features)
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tensor_train_X = torch.tensor(train_features_scaled, dtype=torch.float32).unsqueeze(1).to(device)
tensor_train_Y = torch.tensor(train_target_scaled, dtype=torch.float32).to(device)

tensor_test_X  = torch.tensor(test_features_scaled, dtype=torch.float32).unsqueeze(1).to(device)

dataset = TensorDataset(tensor_train_X, tensor_train_Y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# ---------------------------
# Model (GRU with seq_len = 1)
# ---------------------------
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layers, output_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

model = GRUModel(input_dim=6, hidden_dim=64, layers=2, output_dim=1).to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ---------------------------
# Training
# ---------------------------
for epoch in range(10):
    for Xb, Yb in dataloader:
        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, Yb)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.6f}")

# ---------------------------
# Predict test
# ---------------------------
model.eval()
with torch.no_grad():
    preds_scaled = model(tensor_test_X).cpu().numpy()

# Inverse scale to real OT
preds_real = target_scaler.inverse_transform(preds_scaled)

# ---------------------------
# Save to CSV
# ---------------------------
with open("test_predictions.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "OT"])
    for i, val in enumerate(preds_real):
        writer.writerow([i, float(val[0])])


Epoch 1, Loss: 0.149568
Epoch 2, Loss: 0.090849
Epoch 3, Loss: 0.149568
Epoch 4, Loss: 0.105378
Epoch 5, Loss: 0.137929
Epoch 6, Loss: 0.134468
Epoch 7, Loss: 0.109940
Epoch 8, Loss: 0.135495
Epoch 9, Loss: 0.102699
Epoch 10, Loss: 0.104577
